In [2]:
from pathlib import Path
import pandas as pd

# =========================
# CONFIG CLEAN TESTING DATASET
# =========================

RAW_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/A.Raw_Testing")
CLEAN_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/B.Clean Testing")

ROOMS = ["Controlled Room", "Uncontrolled Room"]

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

TEST_SUBJECTS = [
    "Kanaya",
    "Naila",
    "Nana",
    "Rega",
    "Zaira",
]

RAW_FILES = [f"{i}.Csv" for i in range(1, 10)]
CLEAN_FILES = [f"{i}.csv" for i in range(1, 10)]

SELECTED_COLUMNS = ["Timestamp", "X", "Y", "Z", "Reflectivity"]

expected_total = len(ROOMS) * len(ACTIVITIES) * len(TEST_SUBJECTS) * len(RAW_FILES)

print("Raw testing base directory   :", RAW_BASE_DIR)
print("Clean testing base directory :", CLEAN_BASE_DIR)
print("Raw testing base exists      :", RAW_BASE_DIR.exists())
print("Rooms                        :", ROOMS)
print("Activities                   :", ACTIVITIES)
print("Testing subjects             :", TEST_SUBJECTS)
print("Selected columns             :", SELECTED_COLUMNS)
print("Expected total files         :", expected_total)
print("Output extension             : lowercase .csv")

Raw testing base directory   : /media/spell/Spell-lab/Lidar/A.Raw_Testing
Clean testing base directory : /media/spell/Spell-lab/Lidar/B.Clean Testing
Raw testing base exists      : True
Rooms                        : ['Controlled Room', 'Uncontrolled Room']
Activities                   : ['Bungkuk', 'Duduk', 'Jongkok', 'Jatuh']
Testing subjects             : ['Kanaya', 'Naila', 'Nana', 'Rega', 'Zaira']
Selected columns             : ['Timestamp', 'X', 'Y', 'Z', 'Reflectivity']
Expected total files         : 360
Output extension             : lowercase .csv


In [3]:
# =========================
# GENERATE CLEAN TESTING CSV FILES
# =========================

records = []

for room in ROOMS:
    for activity in ACTIVITIES:
        for subject in TEST_SUBJECTS:
            raw_csv_dir = RAW_BASE_DIR / room / activity / subject / "CSV"
            clean_subject_dir = CLEAN_BASE_DIR / room / activity / subject
            clean_subject_dir.mkdir(parents=True, exist_ok=True)

            for raw_name, clean_name in zip(RAW_FILES, CLEAN_FILES):
                input_path = raw_csv_dir / raw_name
                output_path = clean_subject_dir / clean_name

                status = {
                    "room": room,
                    "activity": activity,
                    "subject": subject,
                    "raw_file": raw_name,
                    "clean_file": clean_name,
                    "input_path": str(input_path),
                    "output_path": str(output_path),
                    "input_exists": input_path.exists(),
                    "success": False,
                    "rows_before": None,
                    "rows_after": None,
                    "dropped_rows": None,
                    "missing_columns": None,
                    "error": None,
                }

                try:
                    if not input_path.exists():
                        status["error"] = "Input file not found"
                        records.append(status)
                        continue

                    # Baca raw CSV
                    df = pd.read_csv(input_path)

                    # Cek kolom wajib
                    missing_cols = [col for col in SELECTED_COLUMNS if col not in df.columns]
                    status["missing_columns"] = missing_cols

                    if missing_cols:
                        status["error"] = f"Missing columns: {missing_cols}"
                        records.append(status)
                        continue

                    status["rows_before"] = len(df)

                    # Ambil hanya kolom penting
                    df_clean = df[SELECTED_COLUMNS].copy()

                    # Convert semua kolom menjadi numeric
                    for col in SELECTED_COLUMNS:
                        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

                    before_drop = len(df_clean)

                    # Drop baris invalid
                    df_clean = df_clean.dropna(subset=SELECTED_COLUMNS).copy()

                    # Sort timestamp agar urutan packet/frame aman
                    df_clean = df_clean.sort_values("Timestamp").reset_index(drop=True)

                    after_drop = len(df_clean)

                    status["rows_after"] = after_drop
                    status["dropped_rows"] = before_drop - after_drop

                    # Simpan sebagai pure CSV standar
                    df_clean.to_csv(
                        output_path,
                        index=False,
                        encoding="utf-8",
                        sep=",",
                        lineterminator="\n"
                    )

                    status["success"] = True

                except Exception as e:
                    status["error"] = str(e)

                records.append(status)

clean_generation_df = pd.DataFrame(records)

print("===== CLEAN TESTING DATASET GENERATION SUMMARY =====")
print("Expected files :", expected_total)
print("Success files  :", int(clean_generation_df["success"].sum()))
print("Failed files   :", int((~clean_generation_df["success"]).sum()))

print("\nDropped rows summary:")
display(clean_generation_df["dropped_rows"].describe())

print("\nFailed files preview:")
display(clean_generation_df[clean_generation_df["success"] == False].head(30))

/tmp/ipykernel_143604/4190910464.py:42: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_143604/4190910464.py:42: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_143604/4190910464.py:42: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_143604/4190910464.py:42: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_143604/4190910464.py:42: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)
/tmp/ipykernel_143604/4190910464.py:42: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df 

===== CLEAN TESTING DATASET GENERATION SUMMARY =====
Expected files : 360
Success files  : 360
Failed files   : 0

Dropped rows summary:


count    360.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: dropped_rows, dtype: float64


Failed files preview:


,room,activity,subject,raw_file,clean_file,input_path,output_path,input_exists,success,rows_before,rows_after,dropped_rows,missing_columns,error


In [4]:
# =========================
# VALIDATE CLEAN TESTING DATASET OUTPUT
# =========================

validation_records = []

for room in ROOMS:
    for activity in ACTIVITIES:
        for subject in TEST_SUBJECTS:
            clean_subject_dir = CLEAN_BASE_DIR / room / activity / subject

            for clean_name in CLEAN_FILES:
                output_path = clean_subject_dir / clean_name

                rec = {
                    "room": room,
                    "activity": activity,
                    "subject": subject,
                    "clean_file": clean_name,
                    "output_path": str(output_path),
                    "exists": output_path.exists(),
                    "can_read": False,
                    "is_lowercase_csv": output_path.name.endswith(".csv"),
                    "file_size_bytes": None,
                    "num_rows": None,
                    "columns": None,
                    "columns_exact_match": False,
                    "has_na": None,
                    "error": None,
                }

                try:
                    if not output_path.exists():
                        rec["error"] = "Output file not found"
                        validation_records.append(rec)
                        continue

                    rec["file_size_bytes"] = output_path.stat().st_size

                    df_check = pd.read_csv(output_path)

                    rec["can_read"] = True
                    rec["num_rows"] = len(df_check)
                    rec["columns"] = list(df_check.columns)
                    rec["columns_exact_match"] = list(df_check.columns) == SELECTED_COLUMNS
                    rec["has_na"] = bool(df_check[SELECTED_COLUMNS].isna().any().any())

                except Exception as e:
                    rec["error"] = str(e)

                validation_records.append(rec)

clean_validation_df = pd.DataFrame(validation_records)

problem_validation_df = clean_validation_df[
    (clean_validation_df["exists"] == False) |
    (clean_validation_df["can_read"] == False) |
    (clean_validation_df["is_lowercase_csv"] == False) |
    (clean_validation_df["columns_exact_match"] == False) |
    (clean_validation_df["has_na"] == True)
]

print("===== CLEAN TESTING DATASET VALIDATION SUMMARY =====")
print("Expected clean files :", expected_total)
print("Existing files       :", int(clean_validation_df["exists"].sum()))
print("Readable files       :", int(clean_validation_df["can_read"].sum()))
print("Lowercase .csv files :", int(clean_validation_df["is_lowercase_csv"].sum()))
print("Exact column match   :", int(clean_validation_df["columns_exact_match"].sum()))
print("Files with NA        :", int((clean_validation_df["has_na"] == True).sum()))
print("Problem files        :", len(problem_validation_df))

print("\nProblem files preview:")
display(problem_validation_df.head(30))

print("\nRow count statistics:")
display(clean_validation_df["num_rows"].describe())

===== CLEAN TESTING DATASET VALIDATION SUMMARY =====
Expected clean files : 360
Existing files       : 360
Readable files       : 360
Lowercase .csv files : 360
Exact column match   : 360
Files with NA        : 0
Problem files        : 0

Problem files preview:


,room,activity,subject,clean_file,output_path,exists,can_read,is_lowercase_csv,file_size_bytes,num_rows,columns,columns_exact_match,has_na,error



Row count statistics:


count    3.600000e+02
mean     1.169727e+06
std      1.460591e+05
min      7.107840e+05
25%      1.097568e+06
50%      1.150848e+06
75%      1.230072e+06
max      1.943712e+06
Name: num_rows, dtype: float64

In [5]:
# =========================
# SUMMARY AND SAVE CLEAN TESTING REPORTS
# =========================

summary_by_subject = (
    clean_validation_df
    .groupby(["room", "activity", "subject"])
    .agg(
        expected_files=("clean_file", "count"),
        existing_files=("exists", "sum"),
        readable_files=("can_read", "sum"),
        lowercase_csv_files=("is_lowercase_csv", "sum"),
        exact_column_files=("columns_exact_match", "sum"),
        files_with_na=("has_na", lambda x: int((x == True).sum())),
        total_rows=("num_rows", "sum"),
    )
    .reset_index()
)

summary_by_subject["missing_files"] = summary_by_subject["expected_files"] - summary_by_subject["existing_files"]
summary_by_subject["completion_percent"] = 100 * summary_by_subject["existing_files"] / summary_by_subject["expected_files"]

summary_by_activity = (
    clean_validation_df
    .groupby(["room", "activity"])
    .agg(
        expected_files=("clean_file", "count"),
        existing_files=("exists", "sum"),
        readable_files=("can_read", "sum"),
        exact_column_files=("columns_exact_match", "sum"),
        total_rows=("num_rows", "sum"),
    )
    .reset_index()
)

REPORT_DIR = CLEAN_BASE_DIR / "_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

generation_report_path = REPORT_DIR / "clean_testing_generation_report.csv"
validation_report_path = REPORT_DIR / "clean_testing_validation_report.csv"
summary_subject_path = REPORT_DIR / "clean_testing_summary_by_room_activity_subject.csv"
summary_activity_path = REPORT_DIR / "clean_testing_summary_by_room_activity.csv"
problem_report_path = REPORT_DIR / "clean_testing_problem_files.csv"

clean_generation_df.to_csv(generation_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
clean_validation_df.to_csv(validation_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
summary_by_subject.to_csv(summary_subject_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
summary_by_activity.to_csv(summary_activity_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")
problem_validation_df.to_csv(problem_report_path, index=False, encoding="utf-8", sep=",", lineterminator="\n")

print("===== SUMMARY BY ROOM & ACTIVITY =====")
display(summary_by_activity)

print("\n===== SUMMARY BY ROOM, ACTIVITY & SUBJECT =====")
display(summary_by_subject)

print("\n===== REPORT SAVED =====")
print("Generation report :", generation_report_path)
print("Validation report :", validation_report_path)
print("Summary subject   :", summary_subject_path)
print("Summary activity  :", summary_activity_path)
print("Problem report    :", problem_report_path)

print("\nFinal status:")
print("Clean success files :", int(clean_generation_df["success"].sum()), "/", expected_total)
print("Validation problems :", len(problem_validation_df))

===== SUMMARY BY ROOM & ACTIVITY =====


,room,activity,expected_files,existing_files,readable_files,exact_column_files,total_rows
0,Controlled Room,Bungkuk,45,45,45,45,53145024
1,Controlled Room,Duduk,45,45,45,45,53793120
2,Controlled Room,Jatuh,45,45,45,45,57667008
3,Controlled Room,Jongkok,45,45,45,45,54249024
4,Uncontrolled Room,Bungkuk,45,45,45,45,49309344
5,Uncontrolled Room,Duduk,45,45,45,45,50117280
6,Uncontrolled Room,Jatuh,45,45,45,45,53764416
7,Uncontrolled Room,Jongkok,45,45,45,45,49056576



===== SUMMARY BY ROOM, ACTIVITY & SUBJECT =====


,room,activity,subject,expected_files,existing_files,readable_files,lowercase_csv_files,exact_column_files,files_with_na,total_rows,missing_files,completion_percent
0,Controlled Room,Bungkuk,Kanaya,9,9,9,9,9,0,10303680,0,100.0
1,Controlled Room,Bungkuk,Naila,9,9,9,9,9,0,10389888,0,100.0
2,Controlled Room,Bungkuk,Nana,9,9,9,9,9,0,10762944,0,100.0
3,Controlled Room,Bungkuk,Rega,9,9,9,9,9,0,10498080,0,100.0
4,Controlled Room,Bungkuk,Zaira,9,9,9,9,9,0,11190432,0,100.0
5,Controlled Room,Duduk,Kanaya,9,9,9,9,9,0,10362624,0,100.0
6,Controlled Room,Duduk,Naila,9,9,9,9,9,0,10188864,0,100.0
7,Controlled Room,Duduk,Nana,9,9,9,9,9,0,10335168,0,100.0
8,Controlled Room,Duduk,Rega,9,9,9,9,9,0,11364384,0,100.0
9,Controlled Room,Duduk,Zaira,9,9,9,9,9,0,11542080,0,100.0



===== REPORT SAVED =====
Generation report : /media/spell/Spell-lab/Lidar/B.Clean Testing/_reports/clean_testing_generation_report.csv
Validation report : /media/spell/Spell-lab/Lidar/B.Clean Testing/_reports/clean_testing_validation_report.csv
Summary subject   : /media/spell/Spell-lab/Lidar/B.Clean Testing/_reports/clean_testing_summary_by_room_activity_subject.csv
Summary activity  : /media/spell/Spell-lab/Lidar/B.Clean Testing/_reports/clean_testing_summary_by_room_activity.csv
Problem report    : /media/spell/Spell-lab/Lidar/B.Clean Testing/_reports/clean_testing_problem_files.csv

Final status:
Clean success files : 360 / 360
Validation problems : 0
